In [3]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# Load in Account A data
df = pd.read_csv("account_a.csv")

# Convert Datetime variable so it is parsed correctly
df["Datetime"] = pd.to_datetime(df["Datetime"], format="mixed", errors="coerce")

# Extract date and hour
df["date_only"] = df["Datetime"].dt.date
df["hour"] = df["Datetime"].dt.hour

# First 2 weeks (14 days) only
first_14_days = sorted(df["date_only"].unique())[:14]
df_14 = df[df["date_only"].isin(first_14_days)].copy()

# Map days and hours indexes
day_map = {day: i + 1 for i, day in enumerate(first_14_days)}
df_14["d"] = df_14["date_only"].map(day_map)
df_14["t"] = df_14["hour"]

# Calculate Price and Quantity
hourly = (
    df_14.groupby(["d", "t"], as_index=False)
    .agg({
        "Quantity": "sum",
        "Settlement.Point.Price.KWH": "mean"
    })
)

Price = {(row["d"], row["t"]): row["Settlement.Point.Price.KWH"] for _, row in hourly.iterrows()}
Quantity = {(row["d"], row["t"]): row["Quantity"] for _, row in hourly.iterrows()}

# Create set indexes
DT = sorted(list(Quantity.keys()))  

# Max batteries for 2 week period
B_max = 10

# Battery cost
C_battery = 11959

# Charging/Discharging rate per battery per hour
r_ch = 5
r_dis = 11.5

m = gp.Model("battery_model_account_a")

# Decision Variables
x_A = m.addVar(vtype=GRB.INTEGER, lb=0, ub=B_max)
e = m.addVars(range(1, B_max + 1), vtype=GRB.BINARY)
y = m.addVars(range(1, B_max + 1), DT, vtype=GRB.BINARY)
charge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
discharge = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
cs = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)
charge_b = m.addVars(range(1, B_max+1), DT, lb=0, vtype=GRB.CONTINUOUS)
discharge_b = m.addVars(range(1, B_max+1), DT, lb=0, vtype=GRB.CONTINUOUS)
grid = m.addVars(DT, vtype=GRB.CONTINUOUS, lb=0)   ## added this 

# Constraints
# sum_b e_A,b = x_A
m.addConstr(
    gp.quicksum(e[b] for b in range(1, B_max + 1)) == x_A,
    name="num_active_batteries"
)

# y_b,A,t,d <= e_A,b
for b in range(1, B_max + 1):
    for d, t in DT:
        m.addConstr(y[b, d, t] <= e[b])


## REMOVED this as this makes it so batteries are used all the time 
# g_{A,t,d} = Quantity_{A,t,d} 
#for d, t in DT: 
#    m.addConstr(discharge[d, t] == Quantity[d, t])

## ADDED makes it so you can use batteries and the grid 
for d, t in DT:
    m.addConstr(grid[d, t] + discharge[d, t] == Quantity[d, t]) 
    
# charging state capacity constraints
for d, t in DT:
    m.addConstr(cs[d, t] <= 13.5 * x_A)
    m.addConstr(cs[d, t] >= 2.7 * x_A)   ## changed this 


# charging/discharging rate limits
for d, t in DT:
    m.addConstr(charge[d, t] <= r_ch * x_A)
    m.addConstr(discharge[d, t] <= r_dis * x_A)

# Charging state across time
DT_sorted = sorted(DT)   

# Charging state constraints
for i, (d, t) in enumerate(DT_sorted):
    if i == 0:
        m.addConstr(
            cs[d, t] == 13.5 * x_A + charge[d, t] - discharge[d, t]
        )
    else:
        prev_d, prev_t = DT_sorted[i - 1]
        m.addConstr(
            cs[d, t] == cs[prev_d, prev_t] + charge[d, t] - discharge[d, t]
        )

# Battery charging constraint
for b in range(1, B_max+1):
    for d, t in DT:
        m.addConstr(
            charge_b[b, d, t] <= r_ch * y[b, d, t],
            name=f"charge_mode_b{b}_d{d}_t{t}"
        )

# Battery discharging constraint
for b in range(1, B_max+1):
    for d, t in DT:
        m.addConstr(
            discharge_b[b, d, t] <= r_dis * (e[b] - y[b, d, t]),
            name=f"discharge_mode_b{b}_d{d}_t{t}"
        )

# Sum of all charge and discharge for each battery must equal the totals
for d, t in DT:
    m.addConstr(
        charge[d, t] == gp.quicksum(charge_b[b, d, t] for b in range(1, B_max+1)),
        name=f"charge_total_d{d}_t{t}"
    )
    
    m.addConstr(
        discharge[d, t] == gp.quicksum(discharge_b[b, d, t] for b in range(1, B_max+1)),
        name=f"discharge_total_d{d}_t{t}"
    )

# Updated obj function to included grid 
energy_cost = gp.quicksum(
    (grid[d, t] + charge[d, t]) * Price[d, t]
    for d, t in DT
)

battery_cost = x_A * C_battery

m.setObjective(energy_cost + battery_cost, GRB.MINIMIZE)

# Optimize
m.optimize() 

original_cost = sum(
    Quantity[d, t] * Price[d, t]
    for d, t in DT
)  

grid_cost = sum(
    grid[d, t].X * Price[d, t]
    for d, t in DT
)

charge_cost = sum(
    charge[d, t].X * Price[d, t]
    for d, t in DT
)

battery_cost_val = x_A.X * C_battery

total_cost = grid_cost + charge_cost + battery_cost_val

savings = original_cost - total_cost

# Print results
if m.status == GRB.OPTIMAL:
    print("\n===== OPTIMAL SOLUTION =====")
    print(f"Number of batteries: {x_A.X:.0f}")
    
    print("\n----- COST COMPARISON -----")
    print(f"Original cost (grid only): ${original_cost:,.2f}")
    print(f"New total cost (with batteries): ${total_cost:,.2f}")
    print(f"  - Grid cost: ${grid_cost:,.2f}")
    print(f"  - Charging cost: ${charge_cost:,.2f}")
    print(f"  - Battery cost: ${battery_cost_val:,.2f}")
    
    print(f"\n Savings: ${savings:,.2f}")
else:
    print("No optimal solution found.") 

for d, t in DT_sorted:
    ch = charge[d, t].X
    dis = discharge[d, t].X
    g = grid[d, t].X

    # Only print meaningful activity
    if ch > 1e-3 or dis > 1e-3:
        if ch > dis:
            action = "CHARGE"
            amount = ch
        else:
            action = "DISCHARGE"
            amount = dis

        print(
            f"Day {d}, Hour {t:02d} | "
            f"{action:10s} | "
            f"Amount: {amount:8.2f} | "
            f"Grid used: {g:8.2f} | "
            f"Price: {Price[d,t]:.4f}"
        )


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Snapdragon(R) X 12-core X1E80100 @ 3.40 GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information